# March Madness 2025 Prediction Model
## Step 1: Environment Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

## Step 2: Data Loading

In [2]:
# Load core data files
m_teams = pd.read_csv('data/MTeams.csv')
w_teams = pd.read_csv('data/WTeams.csv')
m_seasons = pd.read_csv('data/MSeasons.csv')
w_seasons = pd.read_csv('data/WSeasons.csv')

# Game results
m_reg = pd.read_csv('data/MRegularSeasonCompactResults.csv')
w_reg = pd.read_csv('data/WRegularSeasonCompactResults.csv')
m_tourney = pd.read_csv('data/MNCAATourneyCompactResults.csv')
w_tourney = pd.read_csv('data/WNCAATourneyCompactResults.csv')

# Tournament structure
m_seeds = pd.read_csv('data/MNCAATourneySeeds.csv')
w_seeds = pd.read_csv('data/WNCAATourneySeeds.csv')

# Sample submission
submission = pd.read_csv('data/SampleSubmissionStage1.csv')

## Step 3: Data Processing Functions

In [3]:
def process_seeds(seeds_df):
    """Extract numerical seed value from seed string"""
    seeds_df['Seed'] = seeds_df['Seed'].str.extract('(\d+)').astype(int)
    return seeds_df

def calculate_team_stats(results_df, team_id):
    """Calculate basic team statistics from game results"""
    stats = results_df.groupby(['Season', team_id]).agg({
        'WScore': ['mean', 'max', 'min'],
        'LScore': ['mean', 'max', 'min'],
    })
    stats.columns = ['_'.join(col) for col in stats.columns]
    return stats.reset_index()

## Step 4: Feature Engineering

In [4]:
# Process seeds
m_seeds_clean = process_seeds(m_seeds)
w_seeds_clean = process_seeds(w_seeds)

# Calculate regular season stats
m_team_stats = calculate_team_stats(m_reg, 'WTeamID')
w_team_stats = calculate_team_stats(w_reg, 'WTeamID')

## Step 5: Training Data Preparation

In [5]:
def create_tourney_features(tourney_df, seeds_df, team_stats):
    """Create matchup features for tournament games"""
    # Merge team features
    df = tourney_df.merge(seeds_df, 
                        left_on=['Season', 'WTeamID'],
                        right_on=['Season', 'TeamID'])
    df = df.merge(seeds_df,
                left_on=['Season', 'LTeamID'],
                right_on=['Season', 'TeamID'],
                suffixes=('_W', '_L'))
    
    # Merge team stats
    df = df.merge(team_stats,
                left_on=['Season', 'WTeamID'],
                right_on=['Season', 'WTeamID'])
    df = df.merge(team_stats,
                left_on=['Season', 'LTeamID'],
                right_on=['Season', 'WTeamID'],
                suffixes=('_W', '_L'))
    
    # Create differential features
    df['Seed_diff'] = df['Seed_W'] - df['Seed_L']
    df['Score_mean_diff'] = df['WScore_mean_W'] - df['WScore_mean_L']
    return df[['Season', 'Seed_diff', 'Score_mean_diff', 'WScore_mean_W', 'WScore_mean_L']]

# Create training data
m_train = create_tourney_features(m_tourney, m_seeds_clean, m_team_stats)
w_train = create_tourney_features(w_tourney, w_seeds_clean, w_team_stats)
full_train = pd.concat([m_train, w_train])

## Step 6: Model Training

In [6]:
# Split data
X = full_train.drop(['Season'], axis=1)
y = (full_train['Seed_diff'] < 0).astype(int)  # Simple target: lower seed wins

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Initialize models
models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'XGBoost': xgb.XGBClassifier()
}

# Train and evaluate
results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict_proba(X_val_scaled)[:, 1]
    auc = roc_auc_score(y_val, preds)
    results[name] = auc

## Step 7: Generate Predictions

In [8]:
def create_submission_features(sub_df, m_seeds, w_seeds, m_stats, w_stats):
    """Create features for submission format with error handling"""
    # Split ID into components
    sub_df['Season'] = sub_df['ID'].str.split('_').str[0].astype(int)
    sub_df['Team1'] = sub_df['ID'].str.split('_').str[1].astype(int)
    sub_df['Team2'] = sub_df['ID'].str.split('_').str[2].astype(int)
    
    # Initialize feature columns
    features = []
    
    for _, row in sub_df.iterrows():
        season = row['Season']
        t1 = row['Team1']
        t2 = row['Team2']
        
        # Determine gender based on team ID ranges
        gender = 'M' if t1 < 3000 else 'W'  # Men's teams < 3000, Women's >= 3000
        
        # Get seeds with fallback values
        seeds = m_seeds if gender == 'M' else w_seeds
        try:
            seed1 = seeds[(seeds['Season'] == season) & (seeds['TeamID'] == t1)]['Seed'].values[0]
        except IndexError:
            seed1 = 16  # Default to lowest seed if not found
            
        try:
            seed2 = seeds[(seeds['Season'] == season) & (seeds['TeamID'] == t2)]['Seed'].values[0]
        except IndexError:
            seed2 = 16

        # Get stats with fallback to league average
        stats = m_stats if gender == 'M' else w_stats
        try:
            stats1 = stats[(stats['Season'] == season) & (stats['WTeamID'] == t1)].iloc[0]
        except IndexError:
            stats1 = stats[stats['Season'] == season].mean(numeric_only=True)
            
        try:
            stats2 = stats[(stats['Season'] == season) & (stats['WTeamID'] == t2)].iloc[0]
        except IndexError:
            stats2 = stats[stats['Season'] == season].mean(numeric_only=True)

        features.append({
            'ID': row['ID'],
            'Seed_diff': seed1 - seed2,
            'Score_mean_diff': stats1['WScore_mean'] - stats2['WScore_mean'],
            'WScore_mean_W': stats1['WScore_mean'],
            'WScore_mean_L': stats2['WScore_mean']
        })
    
    return pd.DataFrame(features)

# Prepare submission data
submission = pd.read_csv('data/SampleSubmissionStage1.csv')
sub_features = create_submission_features(submission, m_seeds_clean, w_seeds_clean, m_team_stats, w_team_stats)
sub_scaled = scaler.transform(sub_features.drop('ID', axis=1))

# Make predictions with best model
best_model = GradientBoostingClassifier()
best_model.fit(X, y)
submission['Pred'] = best_model.predict_proba(sub_scaled)[:, 1]

# Save submission
submission[['ID', 'Pred']].to_csv('final_submission.csv', index=False)